# `klygo.files` (`paths.py`): Path conversion and inspection

Construct, inspect, and map paths without changing files. A `Path` result is still only a value until another operation creates or modifies an entry.

Run this notebook from top to bottom. It creates only temporary example files.

In [ ]:
!pip install git+https://github.com/IchigoMazone/klygo.git@v2-release

## Prepare sample files

Run this setup again to reset the examples. No Drive mount or manual upload is required.

In [1]:
from tempfile import TemporaryDirectory
from klygo import files

# Run this cell again to replace only this notebook's temporary workspace.
if "tutorial_workspace" in globals():
    tutorial_workspace.cleanup()
tutorial_workspace = TemporaryDirectory(prefix="klygo-files-")
root = files.path(tutorial_workspace.name)
print("Example directory:", root)

Example directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack


In [2]:
(root / "images").mkdir()
(root / "images" / "sample.jpg").write_bytes(b"sample image bytes")
image_path = root / "images" / "sample.jpg"
print("Sample path:", image_path)

Sample path: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.jpg


## Jump to an API

[`path`](#path) · [`join`](#join) · [`normalize`](#normalize) · [`resolve`](#resolve) · [`relative`](#relative) · [`as_posix`](#as_posix) · [`name`](#name) · [`stem`](#stem) · [`extension`](#extension) · [`extensions`](#extensions) · [`compound_extension`](#compound_extension) · [`parent`](#parent) · [`parents`](#parents) · [`is_absolute`](#is_absolute) · [`is_within`](#is_within) · [`common_path`](#common_path) · [`replace_root`](#replace_root) · [`with_name`](#with_name) · [`with_stem`](#with_stem) · [`with_extension`](#with_extension) · [`unique_path`](#unique_path)

<a id="path"></a>

### `files.path`

Convert a string or path object to `pathlib.Path`.

**Signature:** `files.path(value: 'PathInput', expand_user: 'bool' = True) -> 'Path'`

- `value`: Path-like value to transform.
- `expand_user`: Expand a leading ``~`` to the current user's home.

**Returns:** `pathlib.Path` — Native path object; expands home markers but does not resolve ``..``.

In [3]:
print(files.path('images/sample.jpg'))
print(type(files.path('images/sample.jpg')).__name__)

images\sample.jpg
WindowsPath


**Another useful pattern**

In [4]:
home_literal = files.path('~/datasets', expand_user=False)
home_expanded = files.path('~/datasets')
print(home_literal, home_expanded)
print(files.path(root / 'images'))

~\datasets C:\Users\Mine\datasets
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images


**Expected error and recovery**

In [5]:
try:
    files.path(123)
except TypeError as error:
    print("TypeError:", error)

# Fix: pass a string or another os.PathLike[str] value.
print(files.path("123"))

TypeError: value must be str, PathLike, got int
123


<a id="join"></a>

### `files.join`

Join one or more path components.

**Signature:** `files.join(*parts: 'PathInput') -> 'Path'`

- `parts`: One or more path components.

**Returns:** `pathlib.Path` — Joined path.

A later absolute component can reset earlier components. This constructs a path; it does not copy or move any file.

In [6]:
image_path = files.join(root, 'images', 'sample.jpg')
print(image_path)

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.jpg


**Another useful pattern**

In [7]:
dataset_root = files.join(root, 'dataset')
planned_image = files.join(dataset_root, 'images', 'cat.jpg')
print(planned_image, files.exists(planned_image))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\dataset\images\cat.jpg False


**Expected error and recovery**

In [8]:
try:
    files.join()
except ValueError as error:
    print("ValueError:", error)

print(files.join(root, "images"))

ValueError: join() requires at least one path component
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images


<a id="normalize"></a>

### `files.normalize`

Normalize separators and dot components in a path.

**Signature:** `files.normalize(value: 'PathInput') -> 'Path'`

- `value`: Path-like value to transform.

**Returns:** `pathlib.Path` — Lexically normalized path.

This is lexical normalization. It collapses `.` and `..` but does not inspect symlink targets; use `resolve()` for a filesystem-aware absolute path.

In [9]:
print(files.normalize('images/./draft/../sample.jpg'))

images\sample.jpg


**Another useful pattern**

In [10]:
messy = files.join(root, 'images', '..', 'labels', 'cat.txt')
print('Before:', messy)
print('After:', files.normalize(messy))

Before: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\..\labels\cat.txt
After: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\labels\cat.txt


<a id="resolve"></a>

### `files.resolve`

Return an absolute normalized path.

**Signature:** `files.resolve(value: 'PathInput', strict: 'bool' = False) -> 'Path'`

- `value`: Path-like value to transform.
- `strict`: Require the resolved path to exist.

**Returns:** `pathlib.Path` — Absolute normalized path.

In [11]:
print(files.resolve(image_path, strict=True))
try:
    files.resolve(root / 'missing.jpg', strict=True)
except FileNotFoundError as error:
    print(type(error).__name__, '- use strict=False for a prospective path')

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.jpg
FileNotFoundError - use strict=False for a prospective path


**Another useful pattern**

In [12]:
planned = files.resolve(root / 'future' / 'report.json')
print('Absolute path for a file not created yet:', planned)
print('Existing image:', files.resolve(image_path, strict=True))

Absolute path for a file not created yet: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\future\report.json
Existing image: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.jpg


**Expected error and recovery**

In [13]:
try:
    files.resolve(root / "missing.txt", strict=True)
except FileNotFoundError as error:
    print("FileNotFoundError:", error)

print(files.resolve(root / "missing.txt", strict=False))

FileNotFoundError: [WinError 2] The system cannot find the file specified: 'C:\\Users\\Mine\\AppData\\Local\\Temp\\klygo-files-w_g_pack\\missing.txt'
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\missing.txt


<a id="relative"></a>

### `files.relative`

Compute a path relative to a starting directory.

**Signature:** `files.relative(value: 'PathInput', start: 'PathInput' = '.') -> 'Path'`

- `value`: Path-like value to transform.
- `start`: Starting directory used to compute the relative path.

**Returns:** `pathlib.Path` — Path relative to ``start``.

This uses `os.path.relpath`, so the result can begin with `..`. It does not require the target to be inside `start`.

In [14]:
print(files.relative(image_path, root))

images\sample.jpg


**Another useful pattern**

In [15]:
for item in (image_path, root / 'labels' / 'sample.txt'):
    print(files.relative(item, root))

images\sample.jpg
labels\sample.txt


<a id="as_posix"></a>

### `files.as_posix`

Return a path string with forward-slash separators.

**Signature:** `files.as_posix(value: 'PathInput') -> 'str'`

- `value`: Path to represent.

**Returns:** `str` — Path in POSIX separator form without resolving it.

In [16]:
print(files.as_posix(files.join('images', 'sample.jpg')))

images/sample.jpg


**Another useful pattern**

In [17]:
manifest_entry = files.as_posix(files.relative(image_path, root))
print('Portable manifest path:', manifest_entry)

Portable manifest path: images/sample.jpg


<a id="name"></a>

### `files.name`

Return the final path component including its extension.

**Signature:** `files.name(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final path component.

In [18]:
print(files.name('archive.tar.gz'))

archive.tar.gz


**Another useful pattern**

In [19]:
for item in ('cat.jpg', 'archive.tar.gz', '.env'):
    print(item, '->', files.name(item))

cat.jpg -> cat.jpg
archive.tar.gz -> archive.tar.gz
.env -> .env


<a id="stem"></a>

### `files.stem`

Return the final path component without its last extension.

**Signature:** `files.stem(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final component without its last suffix.

Only the final suffix is removed: `archive.tar.gz` has stem `archive.tar`.

In [20]:
print(files.stem('archive.tar.gz'))

archive.tar


**Another useful pattern**

In [21]:
for item in ('cat.jpg', 'archive.tar.gz'):
    print(item, '->', files.stem(item))

cat.jpg -> cat
archive.tar.gz -> archive.tar


<a id="extension"></a>

### `files.extension`

Return the final filename extension.

**Signature:** `files.extension(value: 'PathInput') -> 'str'`

- `value`: Path to inspect or operate on.

**Returns:** `str` — Final suffix, including the leading dot.

Only the final suffix is returned. Use `extensions()` or `compound_extension()` when the full suffix chain matters.

In [22]:
print(files.extension('archive.tar.gz'))

.gz


**Another useful pattern**

In [23]:
for item in ('cat.jpg', 'archive.tar.gz', 'README'):
    print(item, '->', repr(files.extension(item)))

cat.jpg -> '.jpg'
archive.tar.gz -> '.gz'
README -> ''


<a id="extensions"></a>

### `files.extensions`

Return every filename suffix as a tuple.

**Signature:** `files.extensions(value: 'PathInput') -> 'Tuple[str, ...]'`

- `value`: Path-like value to transform.

**Returns:** `tuple[str, ...]` — All suffixes in filename order.

In [24]:
print(files.extensions('archive.tar.gz'))

('.tar', '.gz')


**Another useful pattern**

In [25]:
for item in ('archive.tar.gz', 'photo.jpg'):
    print(item, '->', files.extensions(item))

archive.tar.gz -> ('.tar', '.gz')
photo.jpg -> ('.jpg',)


<a id="compound_extension"></a>

### `files.compound_extension`

Return every filename suffix joined together.

**Signature:** `files.compound_extension(value: 'PathInput') -> 'str'`

- `value`: Path-like value to transform.

**Returns:** `str` — All suffixes joined together.

All suffixes are joined, not just the archive suffixes recognized by `with_extension()`.

In [26]:
print(files.compound_extension('archive.tar.gz'))

.tar.gz


**Another useful pattern**

In [27]:
for item in ('archive.tar.gz', 'backup.sql.gz', 'photo.jpg'):
    print(item, '->', files.compound_extension(item))

archive.tar.gz -> .tar.gz
backup.sql.gz -> .sql.gz
photo.jpg -> .jpg


<a id="parent"></a>

### `files.parent`

Return the immediate parent path.

**Signature:** `files.parent(value: 'PathInput') -> 'Path'`

- `value`: Path to inspect or operate on.

**Returns:** `pathlib.Path` — Immediate parent path.

In [28]:
print(files.parent(image_path))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images


**Another useful pattern**

In [29]:
print('Image directory:', files.parent(image_path))
print('Dataset root:', files.parent(files.parent(image_path)))

Image directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images
Dataset root: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack


<a id="parents"></a>

### `files.parents`

Return every ancestor path from nearest to farthest.

**Signature:** `files.parents(value: 'PathInput') -> 'Tuple[Path, ...]'`

- `value`: Path-like value to transform.

**Returns:** `tuple[pathlib.Path, ...]` — Ancestors from nearest to farthest.

In [30]:
print(files.parents(image_path)[:2])

(WindowsPath('C:/Users/Mine/AppData/Local/Temp/klygo-files-w_g_pack/images'), WindowsPath('C:/Users/Mine/AppData/Local/Temp/klygo-files-w_g_pack'))


**Another useful pattern**

In [31]:
for ancestor in files.parents(image_path)[:3]:
    print(ancestor)

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack
C:\Users\Mine\AppData\Local\Temp


<a id="is_absolute"></a>

### `files.is_absolute`

Check whether a path is absolute.

**Signature:** `files.is_absolute(value: 'PathInput') -> 'bool'`

- `value`: Path-like value to transform.

**Returns:** `bool` — Whether the value is absolute.

In [32]:
print(files.is_absolute(image_path))
print(files.is_absolute('images/sample.jpg'))

True
False


**Another useful pattern**

In [33]:
candidate = files.join(root, 'output', 'result.json')
print(files.is_absolute(candidate))
print(files.is_absolute('output/result.json'))

True
False


<a id="is_within"></a>

### `files.is_within`

Check whether a path is contained by a root directory.

**Signature:** `files.is_within(value: 'PathInput', root: 'PathInput', resolve_paths: 'bool' = True) -> 'bool'`

- `value`: Path-like value to transform.
- `root`: Root directory for the containment check.
- `resolve_paths`: Resolve both paths before testing containment.

**Returns:** `bool` — Whether ``value`` is contained by ``root``.

With the default `resolve_paths=True`, symlinks and `..` are resolved before containment is checked. A path counts as within itself.

In [34]:
print(files.is_within(image_path, root))
print(files.is_within(root.parent / 'other.jpg', root))

True
False


**Another useful pattern**

In [35]:
safe = files.is_within(root / 'images' / 'cat.jpg', root)
outside = files.is_within(root / '..' / 'outside.jpg', root)
print('Inside:', safe, 'outside:', outside)

Inside: True outside: False


<a id="common_path"></a>

### `files.common_path`

Return the longest common parent of multiple paths.

**Signature:** `files.common_path(values: 'Iterable[PathInput]') -> 'Path'`

- `values`: Non-empty collection of paths.

**Returns:** `pathlib.Path` — Longest common parent.

Supply a non-empty iterable. The calculation is lexical; on Windows, paths on different drives can raise `ValueError`.

In [36]:
print(files.common_path([root / 'images' / 'a.jpg', root / 'images' / 'b.jpg']))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images


**Another useful pattern**

In [37]:
dataset_paths = [root / 'images' / 'cat.jpg', root / 'images' / 'dog.jpg']
print('Shared directory:', files.common_path(dataset_paths))

Shared directory: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images


**Expected error and recovery**

In [38]:
try:
    files.common_path([])
except ValueError as error:
    print("ValueError:", error)

print(files.common_path([root / "images", root / "labels"]))

ValueError: common_path() requires at least one path
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack


<a id="replace_root"></a>

### `files.replace_root`

Map a path from one directory tree into another.

**Signature:** `files.replace_root(value: 'PathInput', old_root: 'PathInput', new_root: 'PathInput') -> 'Path'`

- `value`: Path-like value to transform.
- `old_root`: Root removed from the input path.
- `new_root`: Replacement root.

**Returns:** `pathlib.Path` — Mapped path below ``new_root``.

This maps the path's relative location; it does not move data. A value outside `old_root` raises `ValueError`.

In [39]:
print(files.replace_root(image_path, root / 'images', root / 'backup'))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\backup\sample.jpg


**Another useful pattern**

In [40]:
label_path = files.replace_root(image_path, root / 'images', root / 'labels')
label_path = files.with_extension(label_path, '.txt')
print('Matching label path:', label_path)

Matching label path: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\labels\sample.txt


**Expected error and recovery**

In [41]:
try:
    files.replace_root(root.parent / "outside.jpg", root, root / "backup")
except ValueError as error:
    print("ValueError:", error)

ValueError: Path C:\Users\Mine\AppData\Local\Temp\outside.jpg is not inside root C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack


<a id="with_name"></a>

### `files.with_name`

Return a path with a replaced final component.

**Signature:** `files.with_name(value: 'PathInput', new_name: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_name`: Replacement final path component.

**Returns:** `pathlib.Path` — Path with the new final component.

In [42]:
print(files.with_name(image_path, 'renamed.jpg'))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\renamed.jpg


**Another useful pattern**

In [43]:
print(files.with_name(image_path, 'dog.jpg'))
print('Original image stays:', files.exists(image_path))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\dog.jpg
Original image stays: True


**Expected error and recovery**

In [44]:
try:
    files.with_name(image_path, "nested/name.jpg")
except ValueError as error:
    print("ValueError:", error)

print(files.with_name(image_path, "valid-name.jpg"))

ValueError: Invalid name 'nested/name.jpg'
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\valid-name.jpg


<a id="with_stem"></a>

### `files.with_stem`

Return a path with a replaced stem.

**Signature:** `files.with_stem(value: 'PathInput', new_stem: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_stem`: Replacement filename stem.

**Returns:** `pathlib.Path` — Path with the new stem.

In [45]:
print(files.with_stem(image_path, 'renamed'))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\renamed.jpg


**Another useful pattern**

In [46]:
renamed_candidate = files.with_stem(image_path, 'dog')
print(renamed_candidate, files.extension(renamed_candidate))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\dog.jpg .jpg


<a id="with_extension"></a>

### `files.with_extension`

Return a path with a replaced filename extension.

**Signature:** `files.with_extension(value: 'PathInput', new_extension: 'str') -> 'Path'`

- `value`: Path-like value to transform.
- `new_extension`: Replacement extension, with or without a leading dot.

**Returns:** `pathlib.Path` — Path with the new extension.

Known archive suffixes `.tar.gz`, `.tar.xz`, and `.tar.bz2` are replaced together. For other multi-suffix names, only the final suffix is replaced.

In [47]:
print(files.with_extension('archive.tar.gz', '.zip'))
print(files.with_extension(image_path, 'png'))

archive.zip
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.png


**Another useful pattern**

In [48]:
for item in ('archive.tar.gz', 'sample.jpg', 'notes'):
    print(item, '->', files.with_extension(item, 'zip'))

archive.tar.gz -> archive.zip
sample.jpg -> sample.zip
notes -> notes.zip


**Expected error and recovery**

In [49]:
try:
    files.with_extension(image_path, "folder/png")
except ValueError as error:
    print("ValueError:", error)

print(files.with_extension(image_path, "png"))

ValueError: new_extension must not contain path separators
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample.png


<a id="unique_path"></a>

### `files.unique_path`

Return a currently unused path.

**Signature:** `files.unique_path(value: 'PathInput', separator: 'str' = '_', start: 'int' = 1) -> 'Path'`

- `value`: Path-like value to transform.
- `separator`: Text inserted before the numeric suffix.
- `start`: Initial numeric suffix.

**Returns:** `pathlib.Path` — An unused candidate path.

The result is only an unused candidate at that instant. Another process may claim it before you write; `save()` or `copy()` with `overwrite=False` enforces the final no-overwrite check.

In [50]:
print(files.unique_path(image_path))
print(files.unique_path(image_path, separator='-', start=10))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample_1.jpg
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample-10.jpg


**Another useful pattern**

In [51]:
for separator in ('_', '-'):
    print(files.unique_path(image_path, separator=separator, start=2))

C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample_2.jpg
C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\images\sample-2.jpg


**Expected error and recovery**

In [52]:
try:
    files.unique_path(image_path, start=-1)
except ValueError as error:
    print("ValueError:", error)

ValueError: start must be greater than or equal to 0


## Put it together

In [53]:
destination = files.replace_root(image_path, root / 'images', root / 'archive')
print('Planned destination:', files.with_extension(destination, '.png'))
print('Source still exists:', files.exists(image_path))

Planned destination: C:\Users\Mine\AppData\Local\Temp\klygo-files-w_g_pack\archive\sample.png
Source still exists: True


## Remember

- Path helpers in `paths.py` return `pathlib.Path` or string values without renaming, copying, or creating files on disk.
- Next in `klygo.files`: continue with [Document_Files_Filesystem.ipynb](Document_Files_Filesystem.ipynb) ([Colab](https://colab.research.google.com/drive/1mwiGUw9HDH7KbOeJVseVmYt3jVgpRX-a?usp=sharing)) or return to [Document_Files_Overview.ipynb](Document_Files_Overview.ipynb) ([Colab](https://colab.research.google.com/drive/12PhuA8SVe_Q8jfC4f5klsgQifR6Vdf4H?usp=sharing)).
- Related Klygo modules: [`klygo.archive`](../archive/Document_Archive_Overview.ipynb) · [`klygo.config`](../config/Document_Config_Overview.ipynb).